# 개별종목 조합B — RandomForest

`기본모델/02.RandomForest.ipynb`과 같은 `models.random_forest.build_random_forest_baseline`을 가져오고
조합B 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.random_forest import build_random_forest_baseline  # noqa: E402

MODEL_NAME = 'RandomForest'
MODEL_BUILDER = build_random_forest_baseline


In [2]:
# 2. 조합B의 피처 값만 지정합니다.
import json

COMBINATION = 'B'
FEATURE_COLUMNS = (
    'ret_5',
    'ret_20',
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'dist_high_20',
    'dist_high_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합B 피처: ('ret_5', 'ret_20', 'sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'dist_high_20', 'dist_high_60')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,NaN,750,20140217,20140514,0.4403,0.5012,-0.0609,0.3673,0.2548,0.3364
1,2,balanced,980,20150123,20150421,0.3425,0.3978,-0.0553,0.3320,0.3081,0.3269
2,3,balanced,1210,20151228,20160328,0.3505,0.3762,-0.0257,0.3492,0.3455,0.3484
3,4,balanced,1439,20161202,20170228,0.4308,0.4617,-0.0309,0.3969,0.3045,0.3692
4,5,NaN,1669,20171113,20180207,0.3831,0.3901,-0.0070,0.3567,0.2783,0.3331
5,6,balanced,1899,20181024,20190118,0.3833,0.3725,0.0108,0.3823,0.3840,0.3832
6,7,balanced,2129,20190930,20191224,0.4109,0.4781,-0.0673,0.3752,0.3410,0.3735
7,8,balanced,2359,20200902,20201130,0.3694,0.3476,0.0218,0.3688,0.3948,0.3773
8,9,balanced,2589,20210806,20211105,0.3639,0.3916,-0.0278,0.3523,0.2982,0.3356
9,10,balanced,2818,20220714,20221012,0.3489,0.3454,0.0035,0.3474,0.3226,0.3392


,OOS 폴드 평균
accuracy,0.3808
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,-0.0161
macro_f1,0.3636
down_recall,0.3301
core_harmonic_mean,0.3549


재실행 명령: python scripts/run_stock_model_experiment.py
